# Measles M1 – Data Preparation (Group 3, Northern Region)

**From the raw DHIMS2-style exports to a checked district-month panel.** Module M2 models it.

> **SYNTHETIC TEACHING DATA — NOT FOR PUBLIC HEALTH DECISIONS.** Every count, coverage estimate and outbreak in this pack is simulated (seed 20265024). Results must never be presented as Ghana Health Service findings.

**The Director's decision (fictional, January 2025):** *Which three districts in Northern Region should receive priority measles investigation and immunisation outreach planning, and what evidence supports those choices?* Capacity: **two mobile immunisation outreach teams and one surveillance verification team for 14 days.** Evidence window: January 2023 – December 2024.

| Topic | What it does |
|---|---|
| 1.1 | the specification, written before opening the data |
| 1.2 | reshape the long export, resolve duplicate keys, join names to district codes, complete the grid |
| 1.3 | reporting completeness, impossible values, outliers that are errors rather than outbreaks, age consistency |
| 1.4 | denominators, immunisation, school contact, climate, lags and the split; acceptance checks and the `panel_ready.csv` checkpoint |

Output: `measles_panel.rds`. In the web app, choose **Use outputs as a dataset** on the finished run to give Module M2 its input.

## Setting up

The server passes `data_dir` (the uploaded measles pack: `Measles.zip` as it is, or the files of its `training_data` folder) and `out_dir`.

In [ ]:
# Parameters: the web app (papermill) overrides these for every run.
data_dir <- "/data/measles_data"
out_dir  <- "/srv/jobs/outputs"

In [ ]:
dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)
# the dataset may hold the whole pack (Measles/training_data/...) or just the files: find them
hit <- list.files(data_dir, pattern = "^dhims2_measles_export\\.csv$", recursive = TRUE, full.names = TRUE)
if (length(hit) != 1) stop("Expected exactly one dhims2_measles_export.csv in the dataset, found ", length(hit), ". Upload the measles pack (Measles.zip) as the dataset.")
data_dir <- dirname(hit[1])
data_dir
list.files(data_dir)

Load the packages (they are installed in the server image).

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 6, repr.plot.res = 100)
suppressPackageStartupMessages({
  library(tidyverse); library(sf); library(spdep); library(MASS); library(broom)
  library(mgcv); library(scales); library(patchwork)
})
select <- dplyr::select        # MASS has a select() too; make sure dplyr's wins
options(width = 95)
dir.create(out_dir, showWarnings = FALSE)
focus_region <- "Northern"     # the Director's mandate
set.seed(20250101)             # every simulation in Part 2 is reproducible

---

# Part 1 – Data preparation

The terms of reference require that we **start from the raw DHIMS2-style exports and document every join** to the district lookup, reporting rates, population and covariates. `panel_ready.csv` is used only to check our work at the end of Part 1.

---

# Topic 1.1 – The decision question and the table it needs

**Specific objectives**

- Write a modelling specification from the Director's decision, before opening the data.
- Fix the unit of analysis, the outcome and **each** denominator explicitly (measles needs more than one).
- Describe the shape of the finished table, column by column.

**Expected learning outcomes**

By the end of this topic you will be able to:

- State the measles question precisely enough that two analysts would build the same table.
- Justify total, age-specific and child denominators for a routine measles outcome.
- Recognise how far the export is from the table the decision requires.

**The specification**

In [ ]:
# Write the specification before opening a single file.
tibble(item = c("Decision", "Analytic question", "Outcome", "Unit of analysis", "Denominators",
                "Period", "Predictors", "Geographic scope", "Use of the result"),
       decision = c("Which 3 Northern Region districts get priority investigation + outreach planning (Jan 2025)?",
                    "Can MCV1/MCV2 coverage, accumulated susceptibility and school contacts predict measles clusters?",
                    "laboratory-confirmed measles cases (suspected and tested kept for quality checks)",
                    "district-month: one row per district per calendar month",
                    "total population (all-age incidence); <5 and 5-14 populations (age-specific rates); children 0-14 (susceptibles)",
                    "January 2023 to December 2024 (24 months); train to Jun 2024, test Jul-Dec 2024",
                    "last month's confirmed cases, prior-year MCV1/MCV2/dropout/susceptible %, school-contact calendar, season",
                    "Northern Region (16 districts); other regions used as comparators",
                    "rank districts, with uncertainty, for verification and a 14-day outreach microplan")) |> print(n = 10)

**The shape of the finished table**

In [ ]:
# what a single row of the finished table must contain
tibble(column = c("adm2_code", "month", "confirmed", "population", "population_under5 / _5_to_14",
                  "reported", "usable", "mcv1 / mcv2 / dropout / susceptible", "school_contact_index",
                  "cases_lag1", "split"),
       role   = c("the join key: a code, never a name", "first day of the month", "the outcome",
                  "the all-age denominator", "age-specific denominators", "was the district reporting that month?",
                  "reported and free of impossible values", "immunity-gap indicators (annual)",
                  "a predictor known in advance (term calendar)", "a predictor from the previous export",
                  "train or test"),
       why    = c("names are spelled many ways; codes are not", "one value per district per month",
                  "what the model explains and predicts", "turns counts into rates",
                  "children carry the measles burden; all-age rates hide it",
                  "a zero from no reporting is not a zero from no measles",
                  "flagged rows stay in the table but are not trusted",
                  "case counts alone do not show where unprotected children are",
                  "measles spreads through school contact", "last month is the strongest single predictor",
                  "the model is judged on months it never saw")) |> print(n = 12)

**What the export actually looks like**

In [ ]:
raw <- read_csv(file.path(data_dir, "dhims2_measles_export.csv"), show_col_types = FALSE)
dim(raw)
head(raw, 4)
count(raw, Data)                        # four data elements, stacked in one long column
n_distinct(raw$`Organisation unit`); n_distinct(raw$Period)   # 477 spellings for 260 districts

**Check yourself**

1. The Director asks instead "where should we vaccinate?" with no case data. Which lines of the specification change?
2. Why does measles need an age-specific denominator when the malaria table needed only total population?
3. Why is the 14-day, three-team capacity written into the specification rather than left to the end?

---

# Topic 1.2 – From export to district-month table

**Specific objectives**

- Reshape a long, stacked export into one row per district-month.
- Detect and resolve duplicate keys before they multiply rows.
- Join name-based records to official district codes and resolve what does not match.
- Complete the grid so a district-month with no record exists as a row.

**Expected learning outcomes**

By the end of this topic you will be able to:

- Pivot a DHIMS2 export into analysis shape without losing or duplicating records.
- Explain why a join on names is unsafe and demonstrate a cleaned-key join to codes.
- Decide, and document, what to do with a record that matches no official district.
- Produce a complete district-month grid and state what the export did not cover.

**Recode the elements and parse the period**

In [ ]:
# one row per district-month, one column per data element
wide <- raw |>
  mutate(element = recode(Data,
           "Measles suspected cases"     = "suspected",
           "Measles samples tested"      = "tested",
           "Measles lab-confirmed cases" = "confirmed",
           "Measles reported deaths"     = "deaths"),
         month = ymd(paste0(Period, "01")),
         value = suppressWarnings(as.numeric(Value))) |>
  select(source_name = `Organisation unit`, month, element, value)
wide |> count(source_name, month, element) |> filter(n > 1) |> nrow()   # duplicates? must be 0 before pivoting

**Resolve the duplicates**

In [ ]:
dups <- wide |> group_by(source_name, month, element) |> filter(n() > 1) |> arrange(source_name, month, element)
dups |> ungroup() |> distinct(source_name, month, element) |> nrow()     # how many keys are duplicated
dups |> summarise(values_differ = n_distinct(value) > 1, .groups = "drop") |> count(values_differ)  # identical or conflicting?
dups |> head(6)
# a resubmission in DHIMS2 leaves both rows. Identical rows: keep one.
# If they ever differ, keep the larger value (the later submission adds records) and log it.
write_csv(dups, file.path(out_dir, "dq_duplicate_keys.csv"))
wide <- wide |> group_by(source_name, month, element) |> summarise(value = max(value), .groups = "drop")
wide |> count(source_name, month, element) |> filter(n > 1) |> nrow()

**Pivot**

In [ ]:
dm <- wide |> pivot_wider(names_from = element, values_from = value)
dim(dm)
head(dm, 4)
dm |> summarise(across(c(suspected, tested, confirmed, deaths), ~ sum(is.na(.x))))   # blanks after pivoting

**Join to district codes**

In [ ]:
# the join to district codes: on a cleaned key, never on the raw name
lookup <- read_csv(file.path(data_dir, "district_lookup.csv"), show_col_types = FALSE)
head(lookup, 4); nrow(lookup)
clean_key <- function(x) {
  x |> str_to_upper() |> str_replace_all("[^A-Z ]", " ") |> str_squish() |>
    str_remove("( (MUNICIPAL|MUNICIPALITY|METROPOLITAN|METROPOLIS|METRO|DISTRICT|MUN)( ASSEMBLY)?)+$") |>
    str_squish()
}
dm  <- dm |> mutate(key = clean_key(source_name))
lk  <- lookup |> mutate(key = clean_key(source_name)) |> distinct(key, adm2_code)
dm  <- dm |> left_join(lk, by = "key")
dm |> filter(is.na(adm2_code)) |> distinct(source_name) |> nrow()          # names the lookup did not cover
dm |> filter(is.na(adm2_code)) |> distinct(source_name) |> head(8)

**Resolve what is left**

In [ ]:
# resolve the unmatched names against the official district list, then re-join
districts <- st_read(file.path(data_dir, "ghana_districts.gpkg"), quiet = TRUE)
off <- districts |> st_drop_geometry() |> transmute(key = clean_key(adm2_name), adm2_code_off = adm2_code)
stopifnot(!any(duplicated(off$key)))                                         # one key, one district
dm <- dm |> left_join(off, by = "key") |> mutate(adm2_code = coalesce(adm2_code, adm2_code_off)) |> select(-adm2_code_off)
dm |> filter(is.na(adm2_code)) |> distinct(source_name)                     # what is still unmatched, and why
# A name spelled differently in DHIMS2 and the official list (Sagnarigu / Sagnerigu, a Northern district)
# is decided by a person, once, and written down in the code where the next person can read it.
districts |> st_drop_geometry() |> filter(str_detect(adm2_name, "Sagn")) |> select(adm2_code, adm2_name, region)
manual_codes <- c("SAGNARIGU" = "GH172")
dm <- dm |> mutate(adm2_code = coalesce(adm2_code, unname(manual_codes[key])))
dm |> filter(is.na(adm2_code)) |> nrow()
# after mapping to codes, two spellings of the same district could collide: check before dropping names
dm |> count(adm2_code, month) |> filter(n > 1) |> nrow()
dm <- dm |> filter(!is.na(adm2_code)) |> select(-key, -source_name)
dm |> summarise(rows = n(), districts = n_distinct(adm2_code), months = n_distinct(month))
260 * 24                                                                     # the complete grid

**Complete the grid**

In [ ]:
# complete the grid: a district-month that never reported is a missing row, and must exist as a row
grid <- expand_grid(adm2_code = sort(unique(districts$adm2_code)),
                    month = seq(ymd("2023-01-01"), ymd("2024-12-01"), by = "month"))
dm <- grid |> left_join(dm, by = c("adm2_code", "month"))
dim(dm)
dm |> summarise(missing_confirmed = sum(is.na(confirmed)))
nr_codes <- districts |> st_drop_geometry() |> filter(region == focus_region) |> pull(adm2_code)
dm |> filter(adm2_code %in% nr_codes) |> summarise(northern_rows = n(), northern_missing = sum(is.na(confirmed)))

**Figure: districts present each month**

In [ ]:
dm |> mutate(scope = if_else(adm2_code %in% nr_codes, "Northern Region (16)", "Other regions (244)")) |>
  group_by(month, scope) |> summarise(share = mean(!is.na(confirmed)), .groups = "drop") |>
  ggplot(aes(month, share, fill = scope)) + geom_col(position = "dodge") +
  geom_hline(yintercept = 1, colour = "#C9A063", linewidth = 1) +
  scale_fill_manual(values = c("#B2182B", "#010066"), name = NULL) +
  scale_y_continuous(labels = percent) +
  labs(title = "Districts present in the export, by month",
       subtitle = "gold line: every district present. The gap is the missing rows the grid restored.",
       x = NULL, y = "share of districts") + theme_minimal() + theme(legend.position = "bottom")

**Check yourself**

1. The duplicate rows held identical values. What if they had differed by 10 per cent?
2. Why is a cleaned-key match safer than a fuzzy string match, given both are approximate?
3. The Sagnarigu fix is one line. Why does it matter more for Group 3 than for most groups?

---

# Topic 1.3 – Deciding what the model may use

**Specific objectives**

- Distinguish a district-month with no measles from one with no report.
- Use reporting completeness as an explicit, thresholded decision (50% main, 80% sensitivity).
- Detect impossible values from the internal logic of the data elements.
- Separate injected errors from genuine outbreaks, and check age-specific totals against the overall total.

**Expected learning outcomes**

By the end of this topic you will be able to:

- Build a 'reported' indicator and justify the threshold used.
- Write logical consistency checks between related surveillance variables.
- Explain why a median-and-MAD rule over-flags measles outbreaks, and what to use instead.
- Defend a flag-do-not-delete policy to a colleague who wants outliers removed.

**Reporting completeness**

In [ ]:
# reporting completeness: the difference between "no cases" and "no report"
rr <- read_csv(file.path(data_dir, "dhims2_reporting_rates.csv"), show_col_types = FALSE) |>
  transmute(key = clean_key(`Organisation unit`), month = ymd(paste0(Period, "01")),
            reporting_rate = suppressWarnings(as.numeric(Value))) |>
  left_join(lk |> rename(code_lk = adm2_code), by = "key") |>
  left_join(off, by = "key") |>
  mutate(adm2_code = coalesce(code_lk, adm2_code_off, unname(manual_codes[key])))
rr |> filter(is.na(adm2_code)) |> nrow()                                     # unmatched reporting rows: must be 0
rr <- rr |> group_by(adm2_code, month) |> summarise(reporting_rate = max(reporting_rate), .groups = "drop")
dm <- dm |> left_join(rr, by = c("adm2_code", "month"))
summary(dm$reporting_rate)
dm |> mutate(band = cut(reporting_rate, c(-Inf, 50, 80, 95, Inf),
                        labels = c("under 50%", "50-80%", "80-95%", "over 95%"))) |> count(band)

**The reported flag**

In [ ]:
# the reported flag: a month is usable only if the district reported it at all
dm <- dm |> mutate(reported    = !is.na(confirmed) & !is.na(reporting_rate) & reporting_rate >= 50,
                   reported_80 = !is.na(confirmed) & !is.na(reporting_rate) & reporting_rate >= 80)   # sensitivity
dm |> count(reported)
dm |> filter(!reported) |> count(is.na(confirmed), is.na(reporting_rate))
dm |> group_by(adm2_code) |> summarise(months_usable = sum(reported)) |> count(months_usable) |> print(n = 8)
dm |> filter(adm2_code %in% nr_codes) |> summarise(northern_reported_50 = sum(reported), northern_reported_80 = sum(reported_80))

**Impossible values**

In [ ]:
# impossible values: confirmed cannot exceed tested, tested cannot exceed suspected
dm <- dm |> mutate(flag_conf_gt_test = !is.na(confirmed) & !is.na(tested) & confirmed > tested,
                   flag_test_gt_susp = !is.na(tested) & !is.na(suspected) & tested > suspected)
dm |> summarise(conf_gt_test = sum(flag_conf_gt_test), test_gt_susp = sum(flag_test_gt_susp))
dm |> filter(flag_conf_gt_test) |> select(adm2_code, month, suspected, tested, confirmed) |> head(10)

**Outliers: errors, not outbreaks**

Measles comes in outbreaks: a genuine cluster *is* a value far above the district's usual level. The malaria median-and-MAD rule would flag exactly the signal the Director needs. Compare it with a rule based on the internal logic of the data: an injected error is a spike in confirmed cases that is **impossible** given the specimens tested.

In [ ]:
dm <- dm |> group_by(adm2_code) |> mutate(
    med = median(confirmed[reported], na.rm = TRUE),
    mad = mad(confirmed[reported], na.rm = TRUE),
    z_robust = if_else(mad > 0, (confirmed - med) / (1.4826 * mad), NA_real_),
    flag_mad6 = !is.na(z_robust) & abs(z_robust) > 6) |> ungroup() |> select(-med, -mad)
# measles rule: confirmed exceeds tested by 20 or more = impossible spike (data-entry error)
dm <- dm |> mutate(flag_outlier = flag_conf_gt_test & (confirmed - tested) >= 20)
dm |> summarise(mad_rule_flags = sum(flag_mad6), impossible_spike_flags = sum(flag_outlier),
                mad_flags_that_are_possible = sum(flag_mad6 & !flag_conf_gt_test))
dm |> filter(flag_outlier) |> select(adm2_code, month, suspected, tested, confirmed, z_robust) |>
  mutate(z_robust = round(z_robust, 1))
# flag, never delete: the row stays, the model is told not to trust it
dm <- dm |> mutate(usable = reported & !flag_conf_gt_test & !flag_outlier)
dm |> summarise(rows = n(), share = round(mean(usable), 3), usable = sum(usable))

**Age-specific cases against the overall total**

The terms of reference ask us to investigate inconsistencies between age-specific and overall case totals.

In [ ]:
age_cases <- read_csv(file.path(data_dir, "measles_age_cases_monthly.csv"), show_col_types = FALSE) |>
  mutate(month = as.Date(month))
age_cases |> count(adm2_code, month) |> filter(n > 1) |> nrow()            # one row per district-month?
dm <- dm |> left_join(age_cases, by = c("adm2_code", "month")) |>
  mutate(flag_age_parts  = !is.na(total_confirmed) & (cases_under5 + cases_5_to_14 + cases_15plus) != total_confirmed,
         flag_age_mismatch = !is.na(total_confirmed) & !is.na(confirmed) & total_confirmed != confirmed)
dm |> summarise(age_parts_not_summing = sum(flag_age_parts), age_total_vs_export = sum(flag_age_mismatch))
dm |> filter(flag_age_mismatch) |> select(adm2_code, month, tested, confirmed, total_confirmed, flag_outlier, flag_conf_gt_test)
# Age-specific rates later use only usable months where the age total agrees with the export.

**Figure: reporting rates and usable months**

In [ ]:
a <- ggplot(dm, aes(reporting_rate)) + geom_histogram(bins = 40, fill = "#010066") +
  geom_vline(xintercept = c(50, 80), colour = "#C9A063", linewidth = 1, linetype = c(1, 2)) +
  labs(title = "Reporting rate", subtitle = "gold lines: 50% (main) and 80% (sensitivity) cuts", x = "%", y = NULL) + theme_minimal()
b <- dm |> mutate(scope = if_else(adm2_code %in% nr_codes, "Northern", "Other regions")) |>
  group_by(month, scope) |> summarise(usable = mean(usable), .groups = "drop") |>
  ggplot(aes(month, usable, colour = scope)) + geom_line(linewidth = 1) + geom_point() +
  scale_colour_manual(values = c("#B2182B", "#010066"), name = NULL) +
  scale_y_continuous(labels = percent, limits = c(0, 1)) +
  labs(title = "Share of districts usable each month", x = NULL, y = NULL) + theme_minimal() + theme(legend.position = "bottom")
a + b

**Figure: three Northern districts and what the flags mark**

In [ ]:
ex_codes <- c("GH167", "GH164", "GH176")   # Mion, Karaga, Tolon
ex <- dm |> filter(adm2_code %in% ex_codes) |>
  left_join(districts |> st_drop_geometry() |> select(adm2_code, nm = adm2_name), by = "adm2_code")
ggplot(ex, aes(month, confirmed)) +
  geom_line(colour = "grey70") +
  geom_point(aes(colour = case_when(flag_outlier | flag_conf_gt_test ~ "impossible value",
                                    !reported ~ "not reported",
                                    flag_mad6 ~ "outbreak month (MAD rule would delete)",
                                    TRUE ~ "usable")), size = 2) +
  scale_colour_manual(values = c("usable" = "#010066", "impossible value" = "#B2182B",
                                 "not reported" = "#C9A063", "outbreak month (MAD rule would delete)" = "#E08214"), name = NULL) +
  facet_wrap(~ nm, scales = "free_y", ncol = 1) +
  labs(title = "Three Northern districts: what the flags mark",
       subtitle = "flagged points stay in the table; only impossible values are left out of the model",
       x = NULL, y = "confirmed cases") + theme_minimal() + theme(legend.position = "bottom")

**Check yourself**

1. A district reports 0 confirmed measles cases with a reporting rate of 96 per cent. Usable or not? Why?
2. A district has 33 confirmed from 30 tested. Error or outbreak? What would you ask the district before deciding?
3. The threshold moves from 50 to 80 per cent. How many Northern district-months do you lose, and which Director's question becomes harder to answer?

---

# Topic 1.4 – Building and checking the decision table

**Specific objectives**

- Attach total and age-specific denominators and the static district characteristics.
- Attach immunisation (MCV1/MCV2, dropout, susceptibles) and the school-contact calendar, and decide which are legitimate predictors.
- Construct lagged predictors correctly within each district.
- Set a time-based split, verify against written acceptance checks, and only then compare with `panel_ready.csv`.

**Expected learning outcomes**

By the end of this topic you will be able to:

- Compute all-age and age-specific incidence with the correct person-time.
- Explain why the 2024 annual coverage estimate cannot be used to forecast 2024 months.
- Construct lags that do not run across district boundaries, and test that they do not.
- Write an acceptance-check block that must pass before any modelling begins.

**Denominators and district characteristics**

In [ ]:
# denominators and district characteristics
pop     <- read_csv(file.path(data_dir, "population_district.csv"), show_col_types = FALSE)
age_pop <- read_csv(file.path(data_dir, "measles_age_structure.csv"), show_col_types = FALSE)
cov     <- read_csv(file.path(data_dir, "district_covariates.csv"), show_col_types = FALSE)
age_pop |> filter(population_total != pop$population[match(paste(adm2_code, year), paste(pop$adm2_code, pop$year))]) |> nrow()  # do the two population files agree?
panel <- dm |> mutate(year = year(month)) |>
  left_join(pop, by = c("adm2_code", "year")) |>
  left_join(age_pop |> select(adm2_code, year, population_under5, population_5_to_14), by = c("adm2_code", "year")) |>
  left_join(cov |> select(adm2_code, adm2_name, region, eco_zone, elev_m, ndvi, area_km2, n_facilities), by = "adm2_code")
panel |> summarise(missing_pop = sum(is.na(population)), missing_age_pop = sum(is.na(population_under5)), missing_cov = sum(is.na(elev_m)))
panel <- panel |> mutate(incidence_100k   = confirmed / population * 1e5,          # per month
                         population_15plus = population - population_under5 - population_5_to_14,
                         children_0_14    = population_under5 + population_5_to_14,
                         pop_density      = population / area_km2,
                         fac_per_10k      = n_facilities / population * 10000)
panel |> filter(usable) |> summarise(median_incidence_100k = round(median(incidence_100k), 2),
                                     p90 = round(quantile(incidence_100k, 0.9), 2))

**Immunisation and susceptibility**

The file is annual. The 2024 estimate is only finalised after the year ends, so for forecasting 2024 months we carry the **previous year's** values (`*_prev`). The 2024 values are used to describe the January 2025 situation for the Director, not to predict 2024.

In [ ]:
imm <- read_csv(file.path(data_dir, "measles_immunisation.csv"), show_col_types = FALSE)
imm |> summarise(across(-c(adm2_code, year), list(min = min, max = max)))
imm |> filter(mcv2_coverage_pct > mcv1_coverage_pct)   # MCV2 above MCV1 is implausible: a data-quality item for the appendix
imm_prev <- imm |> filter(year == 2023) |>
  transmute(adm2_code, mcv1_prev = mcv1_coverage_pct, mcv2_prev = mcv2_coverage_pct,
            dropout_prev = mcv1_to_mcv2_dropout_pct, susceptible_prev = estimated_susceptible_children_pct)
panel <- panel |> left_join(imm, by = c("adm2_code", "year")) |> left_join(imm_prev, by = "adm2_code") |>
  mutate(susceptible_children_n = estimated_susceptible_children_pct / 100 * children_0_14)
panel |> filter(adm2_code %in% nr_codes, month == ymd("2024-12-01")) |>
  select(adm2_name, mcv1_coverage_pct, mcv2_coverage_pct, mcv1_to_mcv2_dropout_pct, susceptible_children_n) |>
  arrange(-susceptible_children_n) |> mutate(susceptible_children_n = round(susceptible_children_n)) |> print(n = 16)

**School contact and mobility**

In [ ]:
school <- read_csv(file.path(data_dir, "school_contact_calendar.csv"), show_col_types = FALSE) |> mutate(month = as.Date(month))
school |> count(adm2_code, month) |> filter(n > 1) |> nrow()
panel <- panel |> left_join(school, by = c("adm2_code", "month"))
panel |> summarise(missing_school = sum(is.na(school_contact_index)))
# the term calendar is published in advance, so same-month school_contact_index is a legitimate predictor;
# event mobility is an estimate made during the month, so only its lag may be used for forecasting.

**Climate (supporting covariate)**

Kept for comparability with the reference pack. Measles is driven by contact and immunity more than by rainfall, so climate is secondary here.

In [ ]:
clim <- read_csv(file.path(data_dir, "dhims2_climate_app_export.csv"), show_col_types = FALSE) |>
  mutate(element = recode(Data, "CHIRPS precipitation (mm)" = "rain_mm",
                                "ERA5-Land temperature 2m (degC)" = "tmean_c"),
         month = ymd(paste0(Period, "01")), key = clean_key(`Organisation unit`),
         value = suppressWarnings(as.numeric(Value))) |>
  left_join(lk |> rename(code_lk = adm2_code), by = "key") |> left_join(off, by = "key") |>
  mutate(adm2_code = coalesce(code_lk, adm2_code_off, unname(manual_codes[key])))
clim |> filter(is.na(adm2_code)) |> distinct(`Organisation unit`)            # must be empty
clim <- clim |> group_by(adm2_code, month, element) |> summarise(value = mean(value), .groups = "drop") |>
  pivot_wider(names_from = element, values_from = value)
panel <- panel |> left_join(clim, by = c("adm2_code", "month"))
panel |> summarise(missing_rain = sum(is.na(rain_mm)), missing_temp = sum(is.na(tmean_c)))

**Lags, season and the split**

In [ ]:
# lags: group by district first, or the lag runs across the district boundary.
# cases_lag1 is NA when last month was not usable: never impute across districts, never fill with 0.
panel <- panel |> arrange(adm2_code, month) |> group_by(adm2_code) |>
  mutate(cases_lag1    = if_else(lag(usable), lag(confirmed), NA_real_),
         mobility_lag1 = lag(estimated_event_mobility_index, 1),
         rain_lag1 = lag(rain_mm, 1), rain_lag2 = lag(rain_mm, 2), tmean_lag1 = lag(tmean_c, 1)) |> ungroup() |>
  mutate(month_of_year = month(month),
         season = if_else(month_of_year %in% 5:10, "wet", "dry"),
         split  = if_else(month < ymd("2024-07-01"), "train", "test"))
panel |> filter(adm2_code == "GH167") |> select(month, confirmed, usable, cases_lag1, school_contact_index) |> head(4)
panel |> filter(month == ymd("2023-01-01")) |> summarise(lag_across_boundary = sum(!is.na(cases_lag1)))   # must be 0
panel |> count(split) |> mutate(months = n / 260)

**Save the table**

In [ ]:
panel <- panel |> select(adm2_code, adm2_name, region, eco_zone, month, year, month_of_year, season,
                         suspected, tested, confirmed, deaths, reported, reported_80, reporting_rate, usable,
                         flag_conf_gt_test, flag_test_gt_susp, flag_outlier, flag_mad6, flag_age_mismatch,
                         cases_under5, cases_5_to_14, cases_15plus, total_confirmed,
                         population, population_under5, population_5_to_14, population_15plus, children_0_14,
                         incidence_100k, pop_density, fac_per_10k, n_facilities, area_km2, elev_m, ndvi,
                         mcv1_coverage_pct, mcv2_coverage_pct, mcv1_to_mcv2_dropout_pct, estimated_susceptible_children_pct,
                         susceptible_children_n, mcv1_prev, mcv2_prev, dropout_prev, susceptible_prev,
                         school_in_session, school_contact_index, estimated_event_mobility_index, mobility_lag1,
                         rain_mm, tmean_c, rain_lag1, rain_lag2, tmean_lag1, cases_lag1, split)
dim(panel)
glimpse(panel, width = 92)
write_rds(panel, file.path(out_dir, "measles_panel.rds"))
write_csv(panel, file.path(out_dir, "measles_panel.csv"))
list.files(out_dir)

**The acceptance checks**

In [ ]:
# the acceptance check: run this before any modelling, every time the panel is rebuilt
stopifnot(nrow(panel) == 260 * 24)
stopifnot(!any(duplicated(panel[, c("adm2_code", "month")])))
stopifnot(all(!is.na(panel$population)), all(!is.na(panel$population_under5)))
stopifnot(all(panel$confirmed[panel$usable] >= 0))
stopifnot(all(!is.na(panel$mcv1_prev)), all(!is.na(panel$school_contact_index)))
stopifnot(sum(panel$adm2_code %in% nr_codes) == 16 * 24)
tibble(check = c("one row per district-month", "no duplicate keys", "population complete (total + age)",
                 "usable district-months", "districts with 20+ usable months", "Northern usable district-months",
                 "test months"),
       result = c(as.character(nrow(panel)), "0 duplicates", "0 missing",
                  as.character(sum(panel$usable)),
                  as.character(sum(tapply(panel$usable, panel$adm2_code, sum) >= 20)),
                  paste(sum(panel$usable[panel$adm2_code %in% nr_codes]), "of", 16 * 24),
                  "Jul-Dec 2024")) |> print(n = 10)

**Checkpoint: compare with `panel_ready.csv`**

Only now, after building the table ourselves. Explain every difference; do not overwrite our table with the checkpoint.

In [ ]:
ref <- read_csv(file.path(data_dir, "panel_ready.csv"), show_col_types = FALSE) |> mutate(month = as.Date(month))
cmp <- panel |> select(adm2_code, month, suspected, tested, confirmed, reporting_rate, reported, flag_outlier) |>
  inner_join(ref |> select(adm2_code, month, suspected, tested, confirmed, reporting_rate, reported, flag_outlier),
             by = c("adm2_code", "month"), suffix = c("_ours", "_ref"))
same <- function(a, b) (is.na(a) & is.na(b)) | (!is.na(a) & !is.na(b) & a == b)
map_dfr(c("suspected", "tested", "confirmed", "reporting_rate", "reported", "flag_outlier"),
        ~ tibble(column = .x, agree = sum(same(cmp[[paste0(.x, "_ours")]], cmp[[paste0(.x, "_ref")]])),
                 differ = sum(!same(cmp[[paste0(.x, "_ours")]], cmp[[paste0(.x, "_ref")]]))))

**Figure: the signal to explain**

In [ ]:
panel |> filter(usable) |> mutate(scope = if_else(region == focus_region, "Northern", "Other regions")) |>
  group_by(month, scope) |>
  summarise(rate = sum(confirmed) / (sum(population) / 12) * 1e5, .groups = "drop") |>
  ggplot(aes(month, rate, colour = scope)) + geom_line(linewidth = 0.9) + geom_point() +
  geom_vline(xintercept = ymd("2024-06-15"), linetype = 2) +
  scale_colour_manual(values = c("#B2182B", "#010066"), name = NULL) +
  labs(title = "The signal the model has to explain",
       subtitle = "confirmed measles per 100,000 person-years (usable months). Dashed: train | test",
       x = NULL, y = "per 100k person-years") + theme_minimal() + theme(legend.position = "bottom")

**Figure: age-specific burden in the Northern Region, 2024**

In [ ]:
panel |> filter(usable, !flag_age_mismatch, year == 2024, region == focus_region) |>
  group_by(adm2_name) |>
  summarise(`<5 y` = sum(cases_under5) / (sum(population_under5) / 12) * 1e5,
            `5-14 y` = sum(cases_5_to_14) / (sum(population_5_to_14) / 12) * 1e5,
            `15+ y` = sum(cases_15plus) / (sum(population_15plus) / 12) * 1e5,
            raw_cases = sum(confirmed)) |>
  pivot_longer(c(`<5 y`, `5-14 y`, `15+ y`), names_to = "age", values_to = "rate") |>
  mutate(age = factor(age, c("<5 y", "5-14 y", "15+ y"))) |>
  ggplot(aes(reorder(adm2_name, rate), rate, fill = age)) + geom_col(position = "dodge") + coord_flip() +
  scale_fill_manual(values = c("#B2182B", "#E08214", "#C9A063"), name = NULL) +
  labs(title = "Age-specific confirmed rate, Northern Region 2024",
       subtitle = "per 100,000 person-years; children carry the burden, and the ranking differs from raw counts",
       x = NULL, y = NULL) + theme_minimal()

**Figure: school contact and the timing of cases**

In [ ]:
d <- panel |> filter(usable, region == focus_region)
bind_rows(d |> transmute(predictor = "school contact (same month)", x = school_contact_index, inc = incidence_100k),
          d |> transmute(predictor = "event mobility (one month earlier)", x = mobility_lag1, inc = incidence_100k)) |>
  filter(!is.na(x)) |>
  ggplot(aes(x, inc)) + geom_point(alpha = 0.25, colour = "#010066") +
  geom_smooth(method = "gam", formula = y ~ s(x, k = 5), colour = "#C9A063", se = TRUE) +
  facet_wrap(~ predictor, scales = "free_x") +
  labs(title = "Contact and mobility alongside confirmed measles, Northern Region",
       subtitle = "an association to explore, not proof of cause", x = "index (0-1)", y = "confirmed per 100,000 (monthly)") +
  theme_minimal()

**Check yourself**

1. Why does January 2023 have NA in cases_lag1, and what would a 0 have done to the model?
2. Why must the model use `mcv1_prev` and not `mcv1_coverage_pct` to forecast July 2024? When does the 2024 value become legitimate?
3. Mion ranks first on rates but Karaga has more cases. Which should the Director see first, and why?
4. Write one more acceptance check that protects the Northern Region decision specifically.

---

## End of Part 1: the checkpoint

`measles_panel.rds` in `outputs/` has passed the acceptance checks and agrees with `panel_ready.csv`. Data-quality log written: `dq_duplicate_keys.csv`. Part 2 reads the panel back from disk. A model must always start from the saved, checked table, never from whatever happens to be in memory.

## Hand-off

The checked panel and the boundaries, for Module M2.

In [ ]:
# the hand-off folder: what the modelling module needs, so this run can be its dataset
h <- file.path(out_dir, "handoff"); dir.create(h, showWarnings = FALSE, recursive = TRUE)
file.copy(file.path(out_dir, "measles_panel.rds"), file.path(h, "measles_panel.rds"), overwrite = TRUE)
file.copy(file.path(data_dir, "ghana_districts.gpkg"), file.path(h, "ghana_districts.gpkg"), overwrite = TRUE)
list.files(h)

## Export for the web map viewer

Writes the map layers listed in `layers.json`.

In [ ]:
# ---- Export for the web map viewer (added by tools/prepare_notebooks.py) ----
# Each block below is wrapped in safe(): if a step was edited or skipped, the other layers are
# still written. The viewer discovers everything through layers.json.
layers <- list()
safe <- function(label, expr) tryCatch(expr, error = function(e)
  message("Export skipped - ", label, ": ", conditionMessage(e)))
add_layer <- function(...) layers[[length(layers) + 1]] <<- list(...)
wgs <- function(x) sf::st_transform(x, 4326)

# district polygons, in lon/lat, lightly simplified for the web
web_districts <- function(d, tol = 0.002) {
  d <- wgs(d)
  keep <- intersect(c("adm2_code", "adm2_name", "region", "eco_zone"), names(d))
  suppressWarnings(sf::st_simplify(d[, keep], preserveTopology = TRUE, dTolerance = tol))
}
write_geojson <- function(x, name) {
  f <- file.path(out_dir, paste0(name, ".geojson"))
  if (file.exists(f)) file.remove(f)
  x <- dplyr::mutate(x, dplyr::across(dplyr::where(is.numeric), ~ round(.x, 4)))
  sf::st_write(x, f, driver = "GeoJSON", quiet = TRUE, layer_options = "COORDINATE_PRECISION=5")
}
# long district-month table -> one column per month (inc_2024_01, ...) + field list for the slider
widen_months <- function(df, value, prefix) {
  df <- as.data.frame(df)
  d  <- data.frame(adm2_code = df$adm2_code, key = paste0(prefix, "_", format(df$month, "%Y_%m")), v = df[[value]])
  m  <- sort(unique(df$month))
  list(wide   = tidyr::pivot_wider(d, names_from = "key", values_from = "v"),
       fields = lapply(m, function(x) list(field = paste0(prefix, "_", format(x, "%Y_%m")), label = format(x, "%b %Y"))))
}
# x/y/value data frame -> heat-point JSON for the browser + a GeoTIFF for GIS users
heat_export <- function(df, name, crs = 32630, max_pts = 25000) {
  df <- df[is.finite(df$value), c("x", "y", "value")]
  tryCatch(terra::writeRaster(terra::rast(df, type = "xyz", crs = paste0("EPSG:", crs)),
                              file.path(out_dir, paste0(name, ".tif")), overwrite = TRUE),
           error = function(e) message("GeoTIFF skipped for ", name, ": ", conditionMessage(e)))
  if (nrow(df) > max_pts) df <- df[round(seq(1, nrow(df), length.out = max_pts)), ]
  ll <- sf::st_coordinates(wgs(sf::st_as_sf(df, coords = c("x", "y"), crs = crs)))
  jsonlite::write_json(list(points = data.frame(lat = ll[, 2], lng = ll[, 1], value = df$value)),
                       file.path(out_dir, paste0(name, ".heat.json")), digits = 6)
}
im_df  <- function(im) { d <- as.data.frame(im); names(d)[3] <- "value"; d }          # spatstat image
pts_df <- function(s, col) { xy <- sf::st_coordinates(s); data.frame(x = xy[, 1], y = xy[, 2], value = s[[col]]) }

# Measles panel: data quality and immunisation by district, and monthly incidence (usable months only)
safe("measles panel", {
  dq <- panel |> dplyr::group_by(adm2_code) |>
    dplyr::summarise(months_usable = sum(usable), months_missing = sum(is.na(confirmed)),
                     reporting_mean = mean(reporting_rate, na.rm = TRUE),
                     flags_impossible = sum(flag_conf_gt_test | flag_test_gt_susp),
                     age_mismatch = sum(flag_age_mismatch),
                     confirmed_usable = sum(confirmed[usable]),
                     mcv1_2024 = dplyr::first(mcv1_coverage_pct[year == 2024]),
                     dropout_2024 = dplyr::first(mcv1_to_mcv2_dropout_pct[year == 2024]),
                     susceptible_2024 = dplyr::first(susceptible_children_n[year == 2024]), .groups = "drop")
  pn <- panel |> dplyr::mutate(inc = dplyr::if_else(usable, incidence_100k, NA_real_))
  mo <- widen_months(pn, "inc", "inc")
  d  <- web_districts(districts) |> dplyr::left_join(dq, by = "adm2_code") |> dplyr::left_join(mo$wide, by = "adm2_code")
  write_geojson(d, "measles_panel_districts")
  add_layer(id = "quality", title = "Measles panel: data quality and immunisation", kind = "choropleth",
    file = "measles_panel_districts.geojson",
    fields = list(
      list(field = "months_usable",    label = "Usable months (of 24)", type = "numeric"),
      list(field = "reporting_mean",   label = "Mean reporting rate (%)", type = "numeric"),
      list(field = "months_missing",   label = "Months missing from the export", type = "numeric"),
      list(field = "flags_impossible", label = "Months with an impossible value", type = "numeric"),
      list(field = "mcv1_2024",        label = "MCV1 coverage 2024 (%)", type = "numeric"),
      list(field = "dropout_2024",     label = "MCV1-MCV2 dropout 2024 (%)", type = "numeric"),
      list(field = "susceptible_2024", label = "Estimated susceptible children 0-14, 2024", type = "numeric"),
      list(field = "confirmed_usable", label = "Confirmed cases, usable months", type = "numeric")))
  add_layer(id = "monthly", title = "Confirmed measles per 100,000 (monthly)", kind = "choropleth",
    file = "measles_panel_districts.geojson",
    series = list(list(id = "inc", label = "Usable months only", fields = mo$fields))) })

jsonlite::write_json(list(layers = layers), file.path(out_dir, "layers.json"),
                     auto_unbox = TRUE, null = "null", na = "null")
cat("Exported", length(layers), "map layer(s) to", out_dir, "\n")
print(list.files(out_dir))
